
# 12: The PyTorch Training Loop

## The Standard ML Workflow

Every ML project in PyTorch follows the same pattern. Master this loop and you can train *any* model:

```
for epoch in epochs:
    for batch in dataloader:
        1. Forward pass   (compute predictions)
        2. Compute loss   (how wrong?)
        3. Backward pass  (which direction?)
        4. Update weights (step downhill)
        5. Zero gradients (reset for next batch)
```

**This lesson is the payoff for lessons 8–11** — model architecture (L8), loss functions (L4), gradient descent (L3), and autograd (L11) all click together here.

**By the end you will:**
- Write a training loop from memory
- Know what each of the 5 steps does and what breaks without it
- See the difference between synthetic clean data (~100%) and realistic noisy data (~85–90%)


## What You'll Learn
- [ ] Write a complete PyTorch training loop from scratch
- [ ] Explain the roles of loss function, optimizer, and forward/backward passes
- [ ] Compare different optimizers (SGD vs Adam) and their convergence behavior

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 8**: Multi-layer network (the model) | Becomes `nn.Module` — same architecture, cleaner code |
| **Lesson 4**: Cross-entropy loss | Becomes `nn.BCELoss()` or `nn.CrossEntropyLoss()` |
| **Lesson 3**: Gradient descent (manual weight updates) | Becomes `optimizer.step()` — SGD, Adam, etc. |
| **Lesson 11**: Autograd (automatic gradients) | `loss.backward()` computes all gradients in one call |

> This lesson brings every piece together into the standard PyTorch workflow you'll use from now on.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to learn the training loop! 🔄")

## 1. Building Models with nn.Module

In [ ]:
# Simple neural network for classification
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = SimpleNN(input_size=2, hidden_size=8, output_size=1)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")

## 2. DataLoaders: Feeding Data in Batches

In [ ]:
# Generate XOR-like data
np.random.seed(42)
n_samples = 1000

X_np = np.vstack([
    np.random.randn(n_samples//4, 2) * 0.5 + [0, 0],
    np.random.randn(n_samples//4, 2) * 0.5 + [1, 1],
    np.random.randn(n_samples//4, 2) * 0.5 + [0, 1],
    np.random.randn(n_samples//4, 2) * 0.5 + [1, 0],
])
y_np = np.array([0]*(n_samples//4) + [0]*(n_samples//4) + [1]*(n_samples//4) + [1]*(n_samples//4))

X = torch.FloatTensor(X_np)
y = torch.FloatTensor(y_np).reshape(-1, 1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='red', alpha=0.5, label='Class 0')
plt.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='blue', alpha=0.5, label='Class 1')
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('XOR-like Classification Problem')
plt.legend()
plt.show()

## 3. Loss Functions and Optimizers

In [ ]:
print("Common Loss Functions:")
print("-" * 50)
print("Binary: BCELoss, BCEWithLogitsLoss")
print("Multi-class: CrossEntropyLoss, NLLLoss")
print("Regression: MSELoss, L1Loss")

print("\nCommon Optimizers:")
print("-" * 50)
print("SGD: Simple, needs tuning")
print("SGD+momentum: Faster convergence")
print("Adam: Adaptive, works well by default")
print("AdamW: Adam + proper weight decay")

## 4. The Complete Training Loop

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

history = {'loss': [], 'accuracy': []}
n_epochs = 50

print("Training...")
print("=" * 50)

for epoch in range(n_epochs):
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        # 1. Forward pass
        outputs = model(batch_x)
        
        # 2. Compute loss
        loss = criterion(outputs, batch_y)
        
        # 3. Backward pass
        loss.backward()
        
        # 4. Update weights
        optimizer.step()
        
        # 5. Zero gradients
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = epoch_loss / len(dataloader)
    accuracy = correct / total
    history['loss'].append(avg_loss)
    history['accuracy'].append(accuracy)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}, Accuracy = {accuracy:.2%}")

print(f"\n✅ Final: Loss = {history['loss'][-1]:.4f}, Accuracy = {history['accuracy'][-1]:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(history['accuracy'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')

plt.tight_layout()
plt.show()

## 5. Visualize Decision Boundary

In [ ]:
# Set to inference mode
model.train(False)

xx, yy = np.meshgrid(np.linspace(-1, 2, 200), np.linspace(-1, 2, 200))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

with torch.no_grad():
    Z = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdBu', alpha=0.6)
plt.colorbar(label='P(class=1)')
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

plt.scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='red', alpha=0.5, s=20)
plt.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='blue', alpha=0.5, s=20)

plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Learned Decision Boundary')
plt.show()

## 6. Training vs Inference Mode

Some layers behave differently during training vs inference:
- **Dropout**: drops neurons during training only
- **BatchNorm**: uses batch stats during training, running stats during inference

In [ ]:
model_with_dropout = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(16, 1)
)

# Training mode (default)
model_with_dropout.train(True)
print(f"Training mode: {model_with_dropout.training}")

# Inference mode
model_with_dropout.train(False)
print(f"Inference mode: {model_with_dropout.training}")

print("\n⚠️  Always set model.train(False) before inference!")
print("   And use torch.no_grad() to disable gradient computation.")

## 7. Saving and Loading Models

In [ ]:
import os

os.makedirs('../../data', exist_ok=True)

# Save model state dict (recommended approach)
torch.save(model.state_dict(), '../../data/model_weights.pth')
print("Saved model weights")

# Load into a new model
new_model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
new_model.load_state_dict(torch.load('../../data/model_weights.pth'))
new_model.train(False)
print("Loaded model weights")

# Verify
with torch.no_grad():
    test_input = torch.tensor([[0.5, 0.5]])
    original_output = model(test_input)
    loaded_output = new_model(test_input)
    print(f"\nOriginal output: {original_output.item():.4f}")
    print(f"Loaded output: {loaded_output.item():.4f}")
    print("✅ Outputs match!")

## ⚠️ What Can Go Wrong: Shape Mismatches

The most common PyTorch error is **shape mismatch** between model output
and target. Let's see common errors and how to debug them.

In [ ]:
# --- Common Shape Mismatch Errors ---
print("Shape Mismatch Debugging Guide")
print("=" * 55)

# Example 1: Wrong target shape
dummy_model = nn.Linear(2, 1)
dummy_input = torch.randn(4, 2)
dummy_target_wrong = torch.tensor([0, 1, 1, 0])  # Shape: (4,)
dummy_target_right = torch.tensor([0, 1, 1, 0]).float().reshape(-1, 1)  # Shape: (4, 1)

output = dummy_model(dummy_input)
print(f"\n1. Model output shape: {output.shape}")
print(f"   Wrong target shape: {dummy_target_wrong.shape}")
print(f"   Right target shape: {dummy_target_right.shape}")

# Show the error
loss_fn = nn.BCEWithLogitsLoss()
try:
    loss = loss_fn(output, dummy_target_wrong.float())
    print(f"\n   ⚠️  No error, but WRONG: broadcasting silently gives bad results!")
except Exception as e:
    print(f"\n   ❌ Error: {e}")

# The fix
loss = loss_fn(output, dummy_target_right)
print(f"   ✅ Fixed with .reshape(-1, 1): loss = {loss.item():.4f}")

# Example 2: Forgetting to convert types
print(f"\n2. Common type errors:")
int_target = torch.tensor([0, 1, 1, 0])  # int64
print(f"   Integer target dtype: {int_target.dtype}")
float_target = int_target.float()  # float32
print(f"   After .float():       {float_target.dtype}")

print(f"\n🔑 Debugging checklist:")
print(f"   1. ALWAYS print shapes: print(output.shape, target.shape)")
print(f"   2. Use .reshape(-1, 1) or .squeeze() to fix dimensions")
print(f"   3. Use .float() to convert integer targets")
print(f"   4. BCELoss needs float targets, CrossEntropyLoss needs long targets")

## 🌎 Now With Real Data: Noisy Moons

Our XOR-like data was synthetic and clean. Let's try `make_moons` from
sklearn — a noisy, non-trivial classification boundary that shows realistic
training behavior.

In [ ]:
# --- Noisy Moons: realistic classification ---
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split as sk_split

# Generate noisy data
X_moons, y_moons = make_moons(n_samples=500, noise=0.2, random_state=42)
X_m_train, X_m_test, y_m_train, y_m_test = sk_split(
    X_moons, y_moons, test_size=0.2, random_state=42)

# Convert to tensors
X_mt = torch.FloatTensor(X_m_train)
y_mt = torch.FloatTensor(y_m_train).reshape(-1, 1)
X_mtest = torch.FloatTensor(X_m_test)
y_mtest = torch.FloatTensor(y_m_test).reshape(-1, 1)

# Train
moon_model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 1)
)
moon_criterion = nn.BCEWithLogitsLoss()
moon_optimizer = optim.Adam(moon_model.parameters(), lr=0.01)

moon_history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(200):
    # Train
    moon_model.train(True)
    out = moon_model(X_mt)
    loss = moon_criterion(out, y_mt)
    loss.backward()
    moon_optimizer.step()
    moon_optimizer.zero_grad()
    
    # Track metrics
    with torch.no_grad():
        moon_model.train(False)
        train_acc = ((torch.sigmoid(moon_model(X_mt)) > 0.5).float() == y_mt).float().mean()
        test_out = moon_model(X_mtest)
        test_loss = moon_criterion(test_out, y_mtest)
        test_acc = ((torch.sigmoid(test_out) > 0.5).float() == y_mtest).float().mean()
    
    moon_history['train_loss'].append(loss.item())
    moon_history['test_loss'].append(test_loss.item())
    moon_history['train_acc'].append(train_acc.item())
    moon_history['test_acc'].append(test_acc.item())

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Training curves
axes[0].plot(moon_history['train_loss'], label='Train')
axes[0].plot(moon_history['test_loss'], label='Test', linestyle='--')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(moon_history['train_acc'], label='Train')
axes[1].plot(moon_history['test_acc'], label='Test', linestyle='--')
axes[1].set_title('Accuracy')
axes[1].legend()

# Decision boundary
moon_model.train(False)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
with torch.no_grad():
    Z = torch.sigmoid(moon_model(torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]))).numpy().reshape(xx.shape)
axes[2].contourf(xx, yy, Z, levels=10, cmap='RdBu', alpha=0.6)
axes[2].scatter(X_m_test[y_m_test==0, 0], X_m_test[y_m_test==0, 1], c='red', s=20, alpha=0.6)
axes[2].scatter(X_m_test[y_m_test==1, 0], X_m_test[y_m_test==1, 1], c='blue', s=20, alpha=0.6)
axes[2].set_title('Decision Boundary (Test Data)')

plt.tight_layout()
plt.show()

print(f"Final test accuracy: {moon_history['test_acc'][-1]:.1%}")
print(f"\n🔑 Real data differences:")
print(f"   • Accuracy is ~{moon_history['test_acc'][-1]:.0%}, not 100% — noise makes perfection impossible")
print(f"   • Train and test curves should be close (gap = overfitting)")
print(f"   • This is what healthy training looks like!")

## 📝 Check Your Understanding

1. What's the order of operations in a training step?
2. Why must we call `optimizer.zero_grad()`?
3. What's the difference between training and inference mode?
4. Why use `torch.no_grad()` during validation?
5. How do you save and load a trained model?

In [ ]:

# --- Exercise 1: Write the Training Loop from Memory ---
# Fill in the 5 missing steps below. The model, loss, optimizer, and data are set up.
# This is the most important exercise in the module — the training loop is the
# one pattern you'll use in every single PyTorch project.

tiny_model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
tiny_criterion = nn.BCEWithLogitsLoss()
tiny_optimizer = optim.Adam(tiny_model.parameters(), lr=0.01)
tiny_loader = DataLoader(TensorDataset(X[:200], y[:200]), batch_size=16, shuffle=True)

losses = []
tiny_model.train(True)
for epoch in range(15):
    for batch_x, batch_y in tiny_loader:

        # Step 1: Forward pass
        outputs = None  # YOUR CODE: pass batch_x through tiny_model

        # Step 2: Compute loss
        loss = None  # YOUR CODE: use tiny_criterion

        # Step 3: Backward pass — compute gradients
        # YOUR CODE (one line)

        # Step 4: Update weights
        # YOUR CODE (one line)

        # Step 5: Zero gradients for next batch
        # YOUR CODE (one line)

        if loss is not None:
            losses.append(loss.item())

# --- Check ---
assert len(losses) > 10, "Did the loop run? Check your forward pass."
assert losses[0] is not None, "Loss is None — did you complete step 2?"
assert losses[-1] < losses[0], \
    f"Loss should decrease! Got {losses[0]:.3f} → {losses[-1]:.3f}. Check steps 3-5."
print(f"Exercise 1 passed! ✓  (Loss: {losses[0]:.3f} → {losses[-1]:.3f})")

# --- Quick Check: SGD vs Adam ---
# What does Adam optimizer do differently from vanilla SGD?
# a) Adam uses a larger learning rate by default
# b) Adam adapts the learning rate per-parameter based on past gradients
# c) Adam doesn't use gradients at all
# d) Adam only works with specific loss functions

your_answer = None  # Put 'a', 'b', 'c', or 'd'

assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', \
    "Adam tracks running averages of gradients AND squared gradients to adapt learning rates for each parameter!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")


## 🎯 Summary

The PyTorch training loop:
1. **Forward pass**: `outputs = model(inputs)`
2. **Compute loss**: `loss = criterion(outputs, targets)`
3. **Backward pass**: `loss.backward()`
4. **Update weights**: `optimizer.step()`
5. **Zero gradients**: `optimizer.zero_grad()`

Key patterns:
- `model.train(True)` for training, `model.train(False)` for inference
- `torch.no_grad()` during evaluation
- `torch.save()` / `torch.load()` for persistence

**Next up**: Word Embeddings - giving words meaning! →